# Analiza podatkov o prometnih nesrečah v Sloveniji

1. Datoteke moramo najprej združiti v en dataframe objekt in nato počistiti napačne in manjkajoče vrednosti

In [148]:
import pandas as pd
import glob
import os
import numpy as np

path = "PN_datasets"
files = glob.glob(os.path.join(path, "*.csv")) #Get all .csv files from the PN_datasets folder

df_list = []

# For every file(year) add a column in each file that will be the unique accident ID across all files format:
# <year>_<accident_id> 
                  
for file in files:
    temp_df = pd.read_csv(file, sep=';', encoding="windows-1250") # ALL FILES USE THE WINDOWS-1250 ENCODING
    year = "".join(filter(str.isdigit, os.path.basename(filename)))
    temp_df['ID'] = year + "_" + temp_df['ZaporednaStevilkaPN'].astype(str)
    
    df_list.append(temp_df)    
                  
master_df = pd.concat(df_list, ignore_index=True) #merge all files into one dataframe          

#print(master_df.isnull().sum()) # Check the sum of missing value across all atributes


2. Znebiti se moramo manjkajočih vrednosti. Največkrat tako, da dodamo nov razred "NEZNANO" ali podobno. Pri starosti lahko tudi vnesemo mediano vseh starosti ampak to je lahko za nadalnjo analizo moteče

In [149]:
# Cleaning / inserting missing values

# Looking at the data it would seem that people without a UE entry were mostly foreigners by nationality.
# Add a new classification for "TUJINA" in such cases
master_df["UEStalnegaPrebivalisca"] = master_df["UEStalnegaPrebivalisca"].fillna("TUJINA")  
master_df["UEStalnegaPrebivalisca"] = master_df["UEStalnegaPrebivalisca"].replace("NEZNANA OBČ", "TUJINA")

# There is a few missing entries for UEStoritve. Added new class "NEZNANO" since it's not that crucial for our analysis
master_df["UpravnaEnotaStoritve"] = master_df["UpravnaEnotaStoritve"].fillna("NEZNANO")


In [150]:
# When looking at the sum of missing values by attribute it would seem that attributes 
# "Povzrocitelj", "PoskodbaUdelezenca", "VrstaUdelezenca", "Starost" and "UporabaVarnostnegaPasu" are all missing 
# 4300 times so there is obviously something wrong with these entries since no data about the person was gathered
# here we deleted all such entires
master_df.dropna(subset=["Povzrocitelj", "PoskodbaUdelezenca", "VrstaUdelezenca", "UporabaVarnostnegaPasu"], inplace=True)
    
#There was a lot of ages marked as -1 so I decided to input the median age instead (in case we use random algorithms
#that dont do well with missing values)
master_df["Starost"] = master_df["Starost"].replace("NEZNANA OBČ", np.nan) # set missing values to nan before calculating median


#####
# SET MISSING AGE TO MEDIAN??? BEWARE WHEN DOING ANALYSIS
#####

#master_df["Starost"] = master_df["Starost"].fillna(master_df["Starost"].median())


In [151]:
# For people entries wihout a nationality default to "Slovenija" if they aren't labled at "TUJINA" in "UEStalnegaPrebivalisca"
condition = (master_df["Drzavljanstvo"].isna()) & (master_df["UEStalnegaPrebivalisca"] != "TUJINA")
master_df.loc[condition, "Drzavljanstvo"] = "Slovenija"

master_df["Drzavljanstvo"] = master_df["Drzavljanstvo"].fillna("NEZNANO")

# Clean inconsistent missing nationality lables
master_df["Drzavljanstvo"] = master_df["Drzavljanstvo"].replace(["OSEBA IMA NEZNANO DRŽAVLJANSTVO", "BREZ DRŽAVLJANSTVA"], "NEZNANO")

# Lowercase Slovenija -> SLOVENIJA
master_df["Drzavljanstvo"] = master_df["Drzavljanstvo"].replace("Slovenija", "SLOVENIJA")

In [152]:
# Merge "TEKOČ (NORMALEN)" with "NORMALEN" for attribute "StanjePrometa"
master_df["StanjePrometa"] = master_df["StanjePrometa"].replace("NORMALEN", "TEKOČ (NORMALEN)")


# Cap driving licence years to 85 (max was around 90 which is probably wrong??)
master_df['VozniskiStazVLetih'] = master_df['VozniskiStazVLetih'].clip(upper=85)

# The value 0 in VozniskiStazVLetih could be interpreted as an underage person, a passenger, person driving without
# a valid licence, person walking ... It's not clear what the 0 is so I will not input or change it.
# The same goes for "VozniskiStazVMesecih"


# Cap sobriety test results at around 3.00 mg/l (considered to be a deadly dose)
# NOTE: Test result might be 0 even if it wasn't done
# First convert result strings to float
master_df['VrednostAlkotesta'] = master_df['VrednostAlkotesta'].astype(str).str.replace(',', '.')
master_df['VrednostAlkotesta'] = pd.to_numeric(master_df['VrednostAlkotesta'], errors='coerce')
master_df['VrednostAlkotesta'] = master_df['VrednostAlkotesta'].clip(upper=3.50)

#Medical sobriety test results look very wrong and inconsistent. I don't expect to use them in this analysis so
#I will simply delete the column
master_df = master_df.drop(columns=['VrednostStrokovnegaPregleda'])


In [155]:
#with pd.option_context('display.max_rows', None):
#    print(master_df["VrednostStrokovnegaPregleda"].value_counts(dropna=False))


In [157]:
master_df.to_csv("merged_clean.csv")